<a href="https://colab.research.google.com/github/BozNatanium/AI_NLP_labs/blob/main/w2v_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [ ]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [ ]:
w2v_model = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

[==================================================] 100.0% 128.1/128.1MB downloaded
Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)

> 1. glove-twitter-25

Источник данных - твиты (2 млрд) и токены (27 млн)

Примерный объем - 1.2 ГБ

Применение - Анализ тональности, обработка неформальной лексики

> 2. fasttext-wiki-news-subwords-300

Источник данных - Википедия + новости (16 млрд токенов)

Примерный объем - 6.5 ГБ

Применение - Морфологически богатые языки, работа с редкими словами

> 3. word2vec-google-news-300

Источник данных - Гугл новости (около 100 млрд слов)

Примерный объем - 3.3 ГБ

Применение - Классические задачи семантической близости, аналогии






**Базовые операции с векторами**

In [ ]:
# Получаем вектор слова
vector = w2v_model['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [ ]:
# Находим похожие слова
similar_words = w2v_model.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

*Ваш ответ здесь*

а что здесь конкретно надо писать?

**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [6]:
!pip install gensim -q

import gensim
import gensim.downloader as api

# Загрузка модели
print("Загрузка модели glove-twitter-25...")
model = api.load('glove-twitter-25')

print(f"Размер словаря: {len(model.key_to_index)}")
print(f"Размерность векторов: {model.vector_size}")

# Пример нахождения похожих слов
try:
    print("\nСлова, похожие на 'python':")
    for word, score in model.most_similar('python', topn=5):
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'python' отсутствует в словаре этой модели.")

Загрузка модели glove-twitter-25...
[==================================================] 100.0% 104.8/104.8MB downloaded
Размер словаря: 1193514
Размерность векторов: 25

Слова, похожие на 'python':
  matrix: 0.8728
  electronic: 0.8587
  osx: 0.8582
  silicon: 0.8498
  fusion: 0.8461


2. Напишите функцию, которая принимает на вход любое слово и вовращает 10 наиболее близких по вектору слов

In [8]:
def get_top_similar_words(word, model, topn=10):
    """
    Возвращает topn наиболее семантически близких слов.
    Если слово отсутствует в словаре, выводит сообщение.
    """
    try:
        similar = model.most_similar(word, topn=topn)
        return [(w, round(score, 4)) for w, score in similar]
    except KeyError:
        print(f"Слово '{word}' не найдено в словаре модели.")
        return []

# Пример использования
print(get_top_similar_words('snake', model))

[('spider', 0.9408), ('turtle', 0.9222), ('donkey', 0.9183), ('tail', 0.9035), ('frog', 0.9026), ('monkey', 0.9024), ('dinosaur', 0.901), ('hole', 0.892), ('brick', 0.89), ('duck', 0.8899)]


3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [ ]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [10]:
# Установка gensim (если не установлен)
!pip install gensim -q

from gensim.models import Word2Vec

# Кулинарный корпус
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

# Обучение Word2Vec с параметрами:
model = Word2Vec(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    sg=1
)

# Вывод первых 10 слов из словаря
print("Слова в словаре (первые 10):", list(model.wv.key_to_index.keys())[:10])
print(f"Общее количество слов в словаре: {len(model.wv.key_to_index)}")

# Пример проверки
try:
    print("\nСлова, похожие на 'варить':")
    for word, score in model.wv.most_similar('варить', topn=3):
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре.")

Слова в словаре (первые 10): ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']
Общее количество слов в словаре: 65

Слова, похожие на 'варить':
  вино: 0.2398
  ингредиенты: 0.2172
  хлеб: 0.1938


In [ ]:
print(f"Слова в словаре: {list(model.wv.key_to_index.keys())[:10]}...")

4. Проверьте модель

In [ ]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

In [13]:
# Найдите слова, похожие на "духовка"
try:
    similar = model.wv.most_similar('духовка', topn=5)
    print("Слова, похожие на 'духовка':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'духовка' не найдено в словаре")

Слова, похожие на 'духовка':
  ингредиенты: 0.3199
  десерт: 0.3064
  холодильник: 0.2705
  питание: 0.2243
  пирог: 0.2142


In [14]:
# Найдите слова, похожие на "овощи"
try:
    similar = model.wv.most_similar('овощи', topn=5)
    print("Слова, похожие на 'овощи':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'овощи' не найдено в словаре")

Слова, похожие на 'овощи':
  мариновать: 0.2716
  хлеб: 0.2691
  гриль: 0.2546
  фольга: 0.2409
  сахар: 0.2108


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [ ]:
ft_model = FastText(
    sentences=sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

In [17]:
!pip install gensim -q

from gensim.models import FastText

# Кулинарный корпус
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

# Обучение
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

print(f"Размер словаря FastText: {len(ft_model.wv.key_to_index)}")
print("Первые 10 слов словаря:", list(ft_model.wv.key_to_index.keys())[:10])

Размер словаря FastText: 65
Первые 10 слов словаря: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']


6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [18]:
# Поиск похожих слов
print("\n=== Похожие слова для 'варить' ===")
try:
    for word, score in ft_model.wv.most_similar('варить', topn=5):
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("  Слово 'варить' не найдено в словаре.")


=== Похожие слова для 'варить' ===
  жарить: 0.5353
  парить: 0.4805
  месить: 0.3541
  тушить: 0.3405
  специи: 0.2622


In [19]:
print("\n=== Похожие слова для 'духовка' ===")
try:
    for word, score in ft_model.wv.most_similar('духовка', topn=5):
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("  Слово 'духовка' не найдено в словаре.")


=== Похожие слова для 'духовка' ===
  взбивать: 0.4565
  лимон: 0.3561
  салат: 0.3050
  курица: 0.3041
  тост: 0.2944


In [20]:
print("\n=== Похожие слова для 'овощи' ===")
try:
    for word, score in ft_model.wv.most_similar('овощи', topn=5):
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("  Слово 'овощи' не найдено в словаре.")


=== Похожие слова для 'овощи' ===
  жарить: 0.2960
  фольга: 0.2574
  морковь: 0.2297
  соус: 0.2172
  торт: 0.2094


7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [ ]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = model.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Сравниваем для разных слов
compare_models('learning')
compare_models('neural')

In [24]:
!pip install gensim -q

from gensim.models import Word2Vec, FastText

# Кулинарный корпус
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

# Обучение Word2Vec (skip-gram)
print("Обучение Word2Vec...")
word2vec_model = Word2Vec(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    sg=1
)

# Обучение FastText
print("Обучение FastText...")
fasttext_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

# Функция сравнения
def compare_models(word, w2v, ft):
    print(f"\nСлово: '{word}'")
    # Word2Vec
    try:
        w2v_sim = w2v.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_sim]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")
    # FastText
    try:
        ft_sim = ft.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_sim]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Три слова с опечатками
typos = ['варитт', 'духофка', 'оващи']

for typo in typos:
    compare_models(typo, word2vec_model, fasttext_model)

Обучение Word2Vec...
Обучение FastText...

Слово: 'варитт'
  Word2Vec: слово не найдено
  FastText: ['печь', 'варить']

Слово: 'духофка'
  Word2Vec: слово не найдено
  FastText: ['бекон', 'травы']

Слово: 'оващи'
  Word2Vec: слово не найдено
  FastText: ['мариновать', 'овощи']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [ ]:
# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:3]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']


In [ ]:
# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [ ]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [ ]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


8. Сравните схожесть doc_2 и doc_4

In [26]:
!pip install gensim -q

from gensim.models import Doc2Vec
from gensim.models.doc2vec import TaggedDocument

# Сами документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразование в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_docs.append(TaggedDocument(words=tokens, tags=[f"doc_{i}"]))

# Обучение Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов в модели: {len(doc_model.dv.key_to_index)}\n")

# Сравнение схожести doc_2 и doc_4
similarity = doc_model.dv.similarity("doc_2", "doc_4")
print(f"Схожесть doc_2 и doc_4: {similarity:.4f}")
print(f"doc_2: {documents[2]}")
print(f"doc_4: {documents[4]}")

Doc2Vec модель обучена!
Количество документов в модели: 5

Схожесть doc_2 и doc_4: -0.0362
doc_2: python programming for data science
doc_4: computer vision processes images


9. Найдите самый похожий документ на doc_1

In [27]:
most_similar_to_doc1 = doc_model.dv.most_similar("doc_1", topn=1)[0]
print(f"Самый похожий документ на doc_1: {most_similar_to_doc1[0]} (схожесть {most_similar_to_doc1[1]:.4f})")
print(f"Текст doc_1: {documents[1]}")
print(f"Текст найденного: {documents[int(most_similar_to_doc1[0].split('_')[1])]}")

Самый похожий документ на doc_1: doc_0 (схожесть 0.2735)
Текст doc_1: deep learning uses neural networks
Текст найденного: machine learning is interesting


10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [28]:
dimensions = [10, 50, 100]
test_words = ['тушить', 'коптить', 'мариновать']

for dim in dimensions:
    print(f"\n--- Размерность {dim} ---")
    model_dim = Word2Vec(
        sentences=cooking_sentences,
        vector_size=dim,
        window=3,
        min_count=1,
        workers=2,
        sg=1
    )
    for word in test_words:
        try:
            similar = model_dim.wv.most_similar(word, topn=3)
            print(f"  {word}: {[w for w,_ in similar]}")
        except KeyError:
            print(f"  {word}: не найден")


--- Размерность 10 ---
  тушить: ['кофе', 'лимон', 'тесто']
  коптить: не найден
  мариновать: ['вода', 'говядина', 'яичница']

--- Размерность 50 ---
  тушить: ['дрожжи', 'сковорода', 'паста']
  коптить: не найден
  мариновать: ['помидоры', 'овощи', 'десерт']

--- Размерность 100 ---
  тушить: ['тост', 'запекать', 'яблоки']
  коптить: не найден
  мариновать: ['резать', 'суп', 'печь']
